In [ ]:
# Standard library
import os, shutil, warnings, json
from pathlib import Path
from typing import List, Tuple, Dict, Optional

# Numerics & data
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report,
    mean_squared_error, mean_absolute_error,
)
from sklearn.utils.class_weight import compute_class_weight

# Deep learning
import tensorflow as tf
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

warnings.filterwarnings("ignore")
np.random.seed(42)
tf.random.set_seed(42)

print("TensorFlow :", tf.__version__)
print("GPUs found :", tf.config.list_physical_devices("GPU"))


TensorFlow : 2.20.0
GPUs found : [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
# Google Drive Mount
from google.colab import drive
import shutil
from pathlib import Path

# Mount drive
drive.mount('/content/drive')

#  DATA MIGRATION
DRIVE_BASE_PATH = Path('/content/drive/MyDrive/deep-learning/DL4AI_Project_Data')

LOCAL_DATA_PATH = Path('/content/data')

def setup_clean_storage(drive_path: Path, local_path: Path):
    """Copies pre-filtered clean data directly to Colab local storage."""
    if drive_path.exists():
        if not local_path.exists():
            print(f"Copying clean data from {drive_path.name} to local storage...")
            shutil.copytree(drive_path, local_path)
            print("Copy complete.")
        else:
            print(f"Data already exists at {local_path}.")
    else:
        print(f"Error: Could not find {drive_path}. Check your Drive path.")

#  PATHS
VN_CLEAN_DRIVE     = DRIVE_BASE_PATH / "vnindex_clean"
VN_CLEAN_DIR       = LOCAL_DATA_PATH / "vietnam/vnindex_clean"

COMPANIES_CSV    = DRIVE_BASE_PATH / "companies.csv"
OVERVIEW_CSV     = DRIVE_BASE_PATH / "ticker-overview.csv"

MODEL_DIR        = Path("./models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

setup_clean_storage(VN_CLEAN_DRIVE, VN_CLEAN_DIR)

#  GLOBAL HYPER-PARAMETERS
WINDOW_SIZE     = 30          # look-back days
BATCH_SIZE      = 256
EPOCHS          = 40          # early-stopping usually triggers earlier

# Task 3 — signal definition
SIGNAL_HORIZON  = 5           # look H days into the future
BUY_THRESHOLD   = 0.03        # +3% return within H days → buy signal
SELL_THRESHOLD  = -0.03       # −3% return within H days → sell signal

# Task 4 — portfolio config
RISK_FREE_RATE  = 0.05 / 252  # ~5% annual → daily; for Sharpe ratio
N_PORTFOLIO     = 20          # number of stocks in final portfolio
MAX_WEIGHT      = 0.15        # cap any single position at 15%


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Data already exists at /content/data/vietnam/vnindex_clean.


## 1. Shared utilities

Most of these are identical to the Tasks 1/2 notebook — `standardize_columns`, `build_windows`, `per_window_minmax`, `date_split`, etc. Re-defined here so the notebook is self-contained.

In [ ]:
# Column aliases for Vietnamese data
_ALIASES = {
    "Date":   ["date","Date","time","Time","<DTYYYYMMDD>","<Date>","TradingDate"],
    "Open":   ["open","Open","<Open>","<OPEN>"],
    "High":   ["high","High","<High>","<HIGH>"],
    "Low":    ["low","Low","<Low>","<LOW>"],
    "Close":  ["close","Close","<Close>","<CLOSE>"],
    "Volume": ["volume","Volume","<Volume>","<VOL>"],
}

def standardize_columns(df: pd.DataFrame) -> pd.DataFrame:
    rename = {}
    for canon, aliases in _ALIASES.items():
        for a in aliases:
            if a in df.columns and canon not in rename.values():
                rename[a] = canon
                break
    return df.rename(columns=rename)


def load_ticker_csv(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    df = standardize_columns(df)
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    df = df.dropna(subset=["Date"]).sort_values("Date").reset_index(drop=True)
    return df


def date_split(X, y, dates, train_frac=0.70, val_frac=0.15):
    """Global date-based chronological split (same as Task 2)."""
    sorted_dates = np.sort(np.unique(dates))
    n = len(sorted_dates)
    cut_train = sorted_dates[int(n * train_frac)]
    cut_val   = sorted_dates[int(n * (train_frac + val_frac))]
    tr  = dates <  cut_train
    va  = (dates >= cut_train) & (dates < cut_val)
    te  = dates >= cut_val
    return (X[tr], y[tr], dates[tr],
            X[va], y[va], dates[va],
            X[te], y[te], dates[te],
            cut_train, cut_val)


In [ ]:
# Technical indicators
def add_technical_features(df: pd.DataFrame, price_col: str = "Close") -> pd.DataFrame:
    """
    Add classic technical indicators that the project explicitly asks about
    in the Task 3 description (SMA, RSI, MACD, …).

    Each indicator's role:
    -----------------------
    SMA5 / SMA20         : short- and medium-term trend
    EMA12 / EMA26        : reactive trend; basis of MACD
    MACD / MACD_signal   : momentum crossover
    RSI14                : overbought (>70) / oversold (<30) signal
    Vol20                : volatility regime (rolling std of log returns)
    LogVolume            : stabilises heavy-tailed volume
    BBpos                : Bollinger band position (0=at lower, 1=at upper)
    Mom10                : 10-day momentum (price vs price 10 days ago)
    """
    out = df.copy()
    p = out[price_col]

    out["LogRet"]    = np.log(p / p.shift(1))
    out["SMA5"]      = p.rolling(5).mean()
    out["SMA20"]     = p.rolling(20).mean()
    out["EMA12"]     = p.ewm(span=12, adjust=False).mean()
    out["EMA26"]     = p.ewm(span=26, adjust=False).mean()
    out["MACD"]      = out["EMA12"] - out["EMA26"]
    out["MACD_sig"]  = out["MACD"].ewm(span=9, adjust=False).mean()

    delta = p.diff()
    gain  = delta.clip(lower=0).rolling(14).mean()
    loss  = (-delta.clip(upper=0)).rolling(14).mean()
    out["RSI14"]     = 100 - 100 / (1 + gain / loss.replace(0, np.nan))

    out["Vol20"]     = out["LogRet"].rolling(20).std()

    if "Volume" in out.columns:
        out["LogVolume"] = np.log1p(out["Volume"])

    # Bollinger band position
    bb_mid = p.rolling(20).mean()
    bb_std = p.rolling(20).std()
    bb_up, bb_lo = bb_mid + 2*bb_std, bb_mid - 2*bb_std
    out["BBpos"]     = (p - bb_lo) / (bb_up - bb_lo)

    # Momentum
    out["Mom10"]     = p / p.shift(10) - 1

    return out


# Trading Signal Identification

## How we frame the problem

A "buy signal" at day t is **not** "the next day's price will be slightly higher" — that's just noise. We define it as:

> **Buy signal at day t** ⇔ the price will rise by **more than +3%** within the next **5 trading days**.
>
> **Sell signal at day t** ⇔ the price will fall by **more than −3%** within the next **5 trading days**.

This produces meaningful, actionable labels:
- A trader can act on signals (transaction costs ~0.1–0.3% per trade are easily absorbed by a 3% move).
- The 5-day window matches typical short-term swing-trading horizons.
- The `±3%` threshold filters out daily noise on most VN stocks.

## Two separate models

We train **two binary classifiers** rather than one 3-class (buy/hold/sell) model because:
1. Buy and sell signals are **not mutually exclusive** with "hold" in practice — a stock can be a "weak buy" and "weak sell" simultaneously near a decision boundary.
2. Two models give independent probability scores, which is exactly what the project asks for ("a score or a probability").
3. Allows asymmetric thresholds (e.g. only act on >70% buy confidence vs >60% sell confidence).

## Class imbalance

Most days are "do nothing" days (no ±3% move). We handle this with:
- **Class weights** in the loss function (`compute_class_weight`)
- **AUC** as the primary metric (insensitive to threshold choice)
- **Precision/recall** rather than naive accuracy (a model that always predicts "no signal" can hit 90%+ accuracy and be useless)


## A. Signal label generation

For each day t we compute the **forward return** over the next H days using `Close` price:

`forward_return_t = max(High[t+1 .. t+H]) / Close[t] - 1`     (for buy signal)  
`forward_return_t = min(Low[t+1 .. t+H]) / Close[t] - 1`      (for sell signal)

Using `max(High)` for buy and `min(Low)` for sell captures the *best opportunity* within the window — what a trader could actually achieve. Then:
- `buy_label_t = 1` if `forward_return_t > +3%` else 0
- `sell_label_t = 1` if `forward_return_t < −3%` else 0


In [ ]:
def make_signal_labels(
    df: pd.DataFrame,
    horizon: int = SIGNAL_HORIZON,
    buy_thr:  float = BUY_THRESHOLD,
    sell_thr: float = SELL_THRESHOLD,
) -> pd.DataFrame:
    """
    Adds three columns to df:
      - fwd_ret_max  : max return achievable in next `horizon` days (using High)
      - fwd_ret_min  : min return (drawdown) in next `horizon` days  (using Low)
      - buy_label    : 1 if fwd_ret_max > buy_thr  else 0
      - sell_label   : 1 if fwd_ret_min < sell_thr else 0

    Rows where the window extends past the available data are dropped at the end.
    """
    out = df.copy()
    p = out["Close"]

    # Rolling forward max/min over the NEXT `horizon` days
    high = out["High"].shift(-1)   # start at t+1
    low  = out["Low"].shift(-1)
    fwd_max = high.rolling(horizon).max().shift(-(horizon - 1))
    fwd_min = low .rolling(horizon).min().shift(-(horizon - 1))

    out["fwd_ret_max"] = fwd_max / p - 1
    out["fwd_ret_min"] = fwd_min / p - 1

    out["buy_label"]   = (out["fwd_ret_max"] >  buy_thr ).astype(int)
    out["sell_label"]  = (out["fwd_ret_min"] <  sell_thr).astype(int)

    return out.dropna(subset=["fwd_ret_max", "fwd_ret_min"]).reset_index(drop=True)


## B. Window construction for classification

In [ ]:
def build_classification_windows(
    df: pd.DataFrame,
    feature_cols: List[str],
    label_col: str,
    window_size: int,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Returns
    -------
    X     : (N, window_size, n_features)
    y     : (N,) binary labels (the label of the LAST day of each window)
    dates : (N,) anchor date (first day of each window)
    """
    arr_X = df[feature_cols].values.astype(np.float32)
    arr_y = df[label_col].values.astype(np.int8)
    arr_d = df["Date"].values
    n = len(df)

    X_list, y_list, d_list = [], [], []
    for i in range(n - window_size):
        X_list.append(arr_X[i : i + window_size])
        # label = signal evaluated at the END of the window
        y_list.append(arr_y[i + window_size - 1])
        d_list.append(arr_d[i])
    return np.stack(X_list), np.array(y_list), np.array(d_list)


def per_window_zscore(X: np.ndarray) -> np.ndarray:
    """Z-score normalise each feature within each window.

    Why z-score (not min-max) here?
    -------------------------------
    Tasks 1/2 use min-max because we need to invert the scaling to recover
    real prices. For classification we never invert; we just want features
    on a comparable scale. Z-score handles outliers more gracefully and
    keeps the relative shape of indicators like RSI / MACD.
    """
    X_n = X.copy().astype(np.float32)
    for i in range(len(X)):
        for c in range(X.shape[2]):
            col = X[i, :, c]
            mu, sd = col.mean(), col.std()
            sd = sd if sd > 1e-8 else 1.0
            X_n[i, :, c] = (col - mu) / sd
    return X_n


## C. Multi-ticker loader for classification

Same pooled approach as Tasks 1/2: iterate every clean VNINDEX ticker, generate labels, build windows, concatenate.

In [ ]:
def load_all_tickers_for_classification(
    clean_dir: Path,
    feature_cols: List[str],
    label_col: str,             # 'buy_label' or 'sell_label'
    window_size: int,
    horizon: int = SIGNAL_HORIZON,
    buy_thr: float = BUY_THRESHOLD,
    sell_thr: float = SELL_THRESHOLD,
    add_ta: bool = True,
    max_tickers: Optional[int] = None,
    verbose: bool = True,
):
    """Load and window every clean VNINDEX ticker for classification."""
    csv_files = sorted(clean_dir.glob("*.csv"))
    if max_tickers:
        csv_files = csv_files[:max_tickers]

    X_parts, y_parts, d_parts, t_parts = [], [], [], []
    feature_cols_final = None
    skipped = 0

    for i, path in enumerate(csv_files):
        ticker = path.stem.replace("-VNINDEX", "").replace("-History", "")
        try:
            df = load_ticker_csv(path)
            if add_ta:
                df = add_technical_features(df)
            df = make_signal_labels(df, horizon, buy_thr, sell_thr)
            df = df.dropna(subset=feature_cols + [label_col])

            if len(df) < window_size + 10:
                skipped += 1
                continue

            # All tickers must share feature set
            if feature_cols_final is None:
                feature_cols_final = [c for c in feature_cols if c in df.columns]
            else:
                feature_cols_final = [c for c in feature_cols_final if c in df.columns]

            X, y, dates = build_classification_windows(
                df, feature_cols_final, label_col, window_size
            )
            X_parts.append(X)
            y_parts.append(y)
            d_parts.append(dates)
            t_parts.append(np.full(len(X), ticker, dtype=object))

        except Exception as e:
            if verbose:
                print(f"  ⚠ {ticker}: {e}")
            skipped += 1
            continue

        if verbose and (i + 1) % 100 == 0:
            n_so_far = sum(len(a) for a in X_parts)
            print(f"  {i+1}/{len(csv_files)} | {n_so_far:,} windows so far")

    X_all = np.concatenate(X_parts, 0)
    y_all = np.concatenate(y_parts, 0)
    d_all = np.concatenate(d_parts, 0)
    t_all = np.concatenate(t_parts, 0)
    print(f"\nLoaded {len(csv_files) - skipped} tickers | skipped {skipped}")
    print(f"Features ({len(feature_cols_final)}): {feature_cols_final}")
    print(f"X: {X_all.shape}  y: {y_all.shape}  positive rate: {y_all.mean():.2%}")
    return X_all, y_all, d_all, t_all, feature_cols_final


## D. Classifier architecture

Stacked Bi-LSTM model with a **sigmoid output** and **binary cross-entropy** loss. We track AUC and accuracy.

In [ ]:
def build_signal_classifier(
    window_size: int,
    n_features: int,
) -> Model:
    """Bi-LSTM classifier with sigmoid output for binary buy/sell signals."""
    inp = Input(shape=(window_size, n_features), name="input")
    x = layers.Bidirectional(
            layers.LSTM(64, return_sequences=True), name="bilstm")(inp)
    x = layers.Dropout(0.3, name="drop1")(x)
    x = layers.LSTM(32, return_sequences=False, name="lstm2")(x)
    x = layers.Dropout(0.3, name="drop2")(x)
    x = layers.Dense(64, activation="relu", name="dense_shared")(x)
    x = layers.Dropout(0.2)(x)
    out = layers.Dense(1, activation="sigmoid", name="head")(x)

    m = Model(inp, out, name="SignalClassifier")
    m.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss="binary_crossentropy",
        metrics=["accuracy", tf.keras.metrics.AUC(name="auc")],
    )
    return m


CALLBACKS_CLF = [
    EarlyStopping(monitor="val_auc", mode="max",
                  patience=8, restore_best_weights=True),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4, min_lr=1e-6),
]


## E. Classifier evaluation

We report:

- **Accuracy**: overall correctness
- **Precision**: when we say "buy", how often do we really see a 3% rise? (the *reliability* of our signals)
- **Recall**: of all true buy opportunities, how many do we catch?
- **F1**: balance between precision and recall
- **AUC**: threshold-independent ranking quality
- **Confusion matrix**

For trading, **precision matters more than recall**: we'd rather miss some opportunities than act on bad signals (each bad trade costs real money). So we evaluate at multiple confidence thresholds.

In [ ]:
def evaluate_classifier(
    y_true: np.ndarray,
    y_proba: np.ndarray,
    label: str = "",
    thresholds: List[float] = (0.3, 0.4, 0.5, 0.6, 0.7),
):
    """Print accuracy/precision/recall/F1 at multiple thresholds + AUC."""
    auc = roc_auc_score(y_true, y_proba) if len(np.unique(y_true)) > 1 else float("nan")
    print(f"\n══════════ {label} ══════════")
    print(f"  Test windows : {len(y_true):,}")
    print(f"  Positive rate: {y_true.mean():.2%}")
    print(f"  AUC          : {auc:.4f}\n")
    print(f"  {'threshold':>10}  {'acc':>6}  {'prec':>6}  {'recall':>7}  {'F1':>6}  {'n_pos':>7}")
    for thr in thresholds:
        y_pred = (y_proba >= thr).astype(int)
        if y_pred.sum() == 0:
            print(f"  {thr:>10.2f}  {accuracy_score(y_true,y_pred):>6.4f}  "
                  f"{'n/a':>6}  {'n/a':>7}  {'n/a':>6}  {0:>7d}")
            continue
        acc = accuracy_score (y_true, y_pred)
        pre = precision_score(y_true, y_pred, zero_division=0)
        rec = recall_score   (y_true, y_pred, zero_division=0)
        f1  = f1_score       (y_true, y_pred, zero_division=0)
        print(f"  {thr:>10.2f}  {acc:>6.4f}  {pre:>6.4f}  {rec:>7.4f}  "
              f"{f1:>6.4f}  {y_pred.sum():>7d}")

    # Confusion matrix at threshold = 0.5
    cm = confusion_matrix(y_true, (y_proba >= 0.5).astype(int))
    print(f"\n  Confusion matrix (threshold = 0.50):")
    print(f"                 pred=0       pred=1")
    print(f"    true=0    {cm[0,0]:>8d}    {cm[0,1]:>8d}")
    print(f"    true=1    {cm[1,0]:>8d}    {cm[1,1]:>8d}")
    return auc


def trading_simulation(
    y_true: np.ndarray,
    y_proba: np.ndarray,
    fwd_returns: np.ndarray,
    label: str = "",
    threshold: float = 0.6,
    cost_bps: float = 20,
):
    """
    Simulate acting on the signal:
    - When y_proba >= threshold → take a position
    - PnL per trade = fwd_return − round-trip cost (cost_bps basis points)

    Returns a summary dict.
    """
    take = y_proba >= threshold
    n_trades = int(take.sum())
    if n_trades == 0:
        print(f"\n══════════ Trading sim ({label}) — NO TRADES at threshold {threshold} ══════════")
        return {}

    cost = cost_bps / 10_000   # basis points → fraction
    pnl  = fwd_returns[take] - cost
    win_rate = float((pnl > 0).mean())
    avg_pnl  = float(pnl.mean())
    total_pnl = float(pnl.sum())
    sharpe   = float(pnl.mean() / pnl.std() * np.sqrt(252)) if pnl.std() > 0 else float("nan")

    print(f"\n══════════ Trading sim ({label}) — threshold {threshold} ══════════")
    print(f"  N trades   : {n_trades:,}  ({n_trades / len(y_true):.2%} of test windows)")
    print(f"  Win rate   : {win_rate:.2%}")
    print(f"  Avg PnL    : {avg_pnl:+.4%}")
    print(f"  Total PnL  : {total_pnl:+.4%}  (sum of returns, not compounded)")
    print(f"  Sharpe (annualised, very rough): {sharpe:.2f}")
    return dict(n_trades=n_trades, win_rate=win_rate, avg_pnl=avg_pnl,
                 total_pnl=total_pnl, sharpe=sharpe)


# 3.1: Buy signal classifier

In [ ]:
# Feature set: OHLCV + technical indicators
FEATURES_T3 = [
    "Open", "High", "Low", "Close", "Volume",
    "LogRet", "SMA5", "SMA20", "MACD", "MACD_sig",
    "RSI14", "Vol20", "LogVolume", "BBpos", "Mom10",
]

# Load data for BUY classifier
X_b, y_b, dates_b, tickers_b, features_b = load_all_tickers_for_classification(
    VN_CLEAN_DIR, FEATURES_T3, label_col="buy_label",
    window_size=WINDOW_SIZE,
)


  100/348 | 280,675 windows so far
  200/348 | 564,965 windows so far
  300/348 | 865,255 windows so far

Loaded 348 tickers | skipped 0
Features (15): ['Open', 'High', 'Low', 'Close', 'Volume', 'LogRet', 'SMA5', 'SMA20', 'MACD', 'MACD_sig', 'RSI14', 'Vol20', 'LogVolume', 'BBpos', 'Mom10']
X: (994683, 30, 15)  y: (994683,)  positive rate: 49.30%


In [ ]:
# Date split
(X_tr_b, y_tr_b, _, X_va_b, y_va_b, _, X_te_b, y_te_b, _,
 cut_tr_b, cut_va_b) = date_split(X_b, y_b, dates_b)

tickers_te_b = tickers_b[dates_b >= cut_va_b]
print(f"Train: {len(X_tr_b):,}  Val: {len(X_va_b):,}  Test: {len(X_te_b):,}")
print(f"Train positive rate: {y_tr_b.mean():.2%}")
print(f"Test  positive rate: {y_te_b.mean():.2%}")

# Per-window z-score normalisation
X_tr_bn = per_window_zscore(X_tr_b)
X_va_bn = per_window_zscore(X_va_b)
X_te_bn = per_window_zscore(X_te_b)

# Class weights (handle imbalance)
class_w = compute_class_weight("balanced", classes=np.array([0,1]), y=y_tr_b)
class_weight_dict_b = {0: class_w[0], 1: class_w[1]}
print(f"Class weights: {class_weight_dict_b}")


Train: 463,094  Val: 254,652  Test: 276,937
Train positive rate: 49.71%
Test  positive rate: 53.61%
Class weights: {0: np.float64(0.9942632137167591), 1: np.float64(1.00580337168945)}


In [ ]:
import json
import numpy as np

# Convert numpy floats to standard Python floats for JSON serialization
class_weight_dict_b_serializable = {
    key: float(value) for key, value in class_weight_dict_b.items()
}

# Define the save path
save_path = MODEL_DIR / "clcass_weight_buy.json"

# Save the dictionary as a JSON file
with open(save_path, 'w') as f:
    json.dump(class_weight_dict_b_serializable, f, indent=4)

print(f"Class weights for BUY classifier saved to: {save_path}")


Class weights for BUY classifier saved to: models/class_weight_buy.json


In [ ]:
# Train BUY classifier
model_buy = build_signal_classifier(WINDOW_SIZE, n_features=len(features_b))
model_buy.summary()

history_buy = model_buy.fit(
    X_tr_bn, y_tr_b,
    validation_data=(X_va_bn, y_va_b),
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    class_weight=class_weight_dict_b,
    callbacks=CALLBACKS_CLF, verbose=2,
)

model_buy.save(MODEL_DIR / "vietnam_buy_classifier.keras")

Model: "SignalClassifier"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input (InputLayer)              │ (None, 30, 15)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bilstm (Bidirectional)          │ (None, 30, 128)        │        40,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ drop1 (Dropout)                 │ (None, 30, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm2 (LSTM)                    │ (None, 32)             │        20,608 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ drop2 (Dropout)                 │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_shared (Dense)            │ (None, 64)             │         2,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ head (Dense)                    │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 63,745 (249.00 KB)

 Trainable params: 63,745 (249.00 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/40
1809/1809 - 31s - 17ms/step - accuracy: 0.5901 - auc: 0.6248 - loss: 0.6687 - val_accuracy: 0.5954 - val_auc: 0.6167 - val_loss: 0.6654 - learning_rate: 0.0010
Epoch 2/40
1809/1809 - 28s - 15ms/step - accuracy: 0.6153 - auc: 0.6603 - loss: 0.6521 - val_accuracy: 0.6059 - val_auc: 0.6279 - val_loss: 0.6597 - learning_rate: 0.0010
Epoch 3/40
1809/1809 - 27s - 15ms/step - accuracy: 0.6252 - auc: 0.6738 - loss: 0.6444 - val_accuracy: 0.6075 - val_auc: 0.6301 - val_loss: 0.6591 - learning_rate: 0.0010
Epoch 4/40
1809/1809 - 27s - 15ms/step - accuracy: 0.6318 - auc: 0.6838 - loss: 0.6382 - val_accuracy: 0.6082 - val_auc: 0.6309 - val_loss: 0.6598 - learning_rate: 0.0010
Epoch 5/40
1809/1809 - 27s - 15ms/step - accuracy: 0.6379 - auc: 0.6920 - loss: 0.6329 - val_accuracy: 0.6043 - val_auc: 0.6273 - val_loss: 0.6632 - learning_rate: 0.0010
Epoch 6/40
1809/1809 - 31s - 17ms/step - accuracy: 0.6437 - auc: 0.6999 - loss: 0.6277 - val_accuracy: 0.6048 - val_auc: 0.6254 - val_loss: 0.665

In [ ]:
# Evaluate BUY classifier
y_proba_buy = model_buy.predict(X_te_bn, verbose=0).flatten()

evaluate_classifier(y_te_b, y_proba_buy, label="Task 3.1 — BUY classifier")

fwd_returns_buy = get_fwd_returns_for_test(VN_CLEAN_DIR, "buy_label", "fwd_ret_max")
trading_simulation(y_te_b, y_proba_buy, fwd_returns_buy,
                   label="BUY", threshold=0.6, cost_bps=20)



══════════ Task 3.1 — BUY classifier ══════════
  Test windows : 276,937
  Positive rate: 53.61%
  AUC          : 0.6393

   threshold     acc    prec   recall      F1    n_pos
        0.30  0.5716  0.5595   0.9435  0.7025   250332
        0.40  0.6008  0.5972   0.7843  0.6781   194980
        0.50  0.5909  0.6459   0.5242  0.5787   120472
        0.60  0.5532  0.6985   0.2929  0.4127    62246
        0.70  0.5127  0.7570   0.1338  0.2274    26243

  Confusion matrix (threshold = 0.50):
                 pred=0       pred=1
    true=0       85827       42656
    true=1       70638       77816

══════════ Trading sim (BUY) — threshold 0.6 ══════════
  N trades   : 62,246  (22.48% of test windows)
  Win rate   : 91.27%
  Avg PnL    : +7.0488%
  Total PnL  : +438757.5195%  (sum of returns, not compounded)
  Sharpe (annualised, very rough): 15.40


{'n_trades': 62246,
 'win_rate': 0.9126530218809241,
 'avg_pnl': 0.07048766314983368,
 'total_pnl': 4387.5751953125,
 'sharpe': 15.404599340343411}

# 3.2: Sell signal classifier

Same architecture and pipeline, but with `sell_label` as the target.

In [ ]:
# Feature set: OHLCV + technical indicators
FEATURES_T3 = [
    "Open", "High", "Low", "Close", "Volume",
    "LogRet", "SMA5", "SMA20", "MACD", "MACD_sig",
    "RSI14", "Vol20", "LogVolume", "BBpos", "Mom10",
]

In [ ]:
# Load data for SELL classifier
X_s, y_s, dates_s, tickers_s, features_s = load_all_tickers_for_classification(
    VN_CLEAN_DIR, FEATURES_T3, label_col="sell_label",
    window_size=WINDOW_SIZE,
)

(X_tr_s, y_tr_s, _, X_va_s, y_va_s, _, X_te_s, y_te_s, _,
 cut_tr_s, cut_va_s) = date_split(X_s, y_s, dates_s)

tickers_te_s = tickers_s[dates_s >= cut_va_s]

X_tr_sn = per_window_zscore(X_tr_s)
X_va_sn = per_window_zscore(X_va_s)
X_te_sn = per_window_zscore(X_te_s)

class_w = compute_class_weight("balanced", classes=np.array([0,1]), y=y_tr_s)
class_weight_dict_s = {0: class_w[0], 1: class_w[1]}
print(f"SELL class weights: {class_weight_dict_s}")

# Convert numpy floats to standard Python floats for JSON serialization
class_weight_dict_s_serializable = {
    key: float(value) for key, value in class_weight_dict_s.items()
}

# Define the save path
save_path = MODEL_DIR / "clcass_weight_buy.json"

# Save the dictionary as a JSON file
with open(save_path, 'w') as f:
    json.dump(class_weight_dict_s_serializable, f, indent=4)

print(f"Class weights for BUY classifier saved to: {save_path}")


  100/348 | 280,675 windows so far
  200/348 | 564,965 windows so far
  300/348 | 865,255 windows so far

Loaded 348 tickers | skipped 0
Features (15): ['Open', 'High', 'Low', 'Close', 'Volume', 'LogRet', 'SMA5', 'SMA20', 'MACD', 'MACD_sig', 'RSI14', 'Vol20', 'LogVolume', 'BBpos', 'Mom10']
X: (994683, 30, 15)  y: (994683,)  positive rate: 53.19%
SELL class weights: {0: np.float64(1.0778350851386704), 1: np.float64(0.9326493950086198)}
Class weights for BUY classifier saved to: models/clcass_weight_buy.json


In [ ]:
model_sell1 = build_signal_classifier(WINDOW_SIZE, n_features=len(features_s))

history_sell1 = model_sell1.fit(
    X_tr_sn, y_tr_s,
    validation_data=(X_va_sn, y_va_s),
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    class_weight=class_weight_dict_s,
    callbacks=CALLBACKS_CLF, verbose=2,
)
model_sell1.save(MODEL_DIR / "vietnam_sell_classifier.keras")

Epoch 1/40
1809/1809 - 36s - 20ms/step - accuracy: 0.5911 - auc: 0.6252 - loss: 0.6684 - val_accuracy: 0.6010 - val_auc: 0.6406 - val_loss: 0.6626 - learning_rate: 0.0010
Epoch 2/40
1809/1809 - 30s - 16ms/step - accuracy: 0.6259 - auc: 0.6734 - loss: 0.6460 - val_accuracy: 0.6125 - val_auc: 0.6580 - val_loss: 0.6548 - learning_rate: 0.0010
Epoch 3/40
1809/1809 - 31s - 17ms/step - accuracy: 0.6368 - auc: 0.6884 - loss: 0.6369 - val_accuracy: 0.6172 - val_auc: 0.6632 - val_loss: 0.6528 - learning_rate: 0.0010
Epoch 4/40
1809/1809 - 31s - 17ms/step - accuracy: 0.6436 - auc: 0.6984 - loss: 0.6303 - val_accuracy: 0.6172 - val_auc: 0.6632 - val_loss: 0.6562 - learning_rate: 0.0010
Epoch 5/40
1809/1809 - 29s - 16ms/step - accuracy: 0.6508 - auc: 0.7078 - loss: 0.6238 - val_accuracy: 0.6161 - val_auc: 0.6621 - val_loss: 0.6549 - learning_rate: 0.0010
Epoch 6/40
1809/1809 - 30s - 17ms/step - accuracy: 0.6574 - auc: 0.7164 - loss: 0.6176 - val_accuracy: 0.6148 - val_auc: 0.6607 - val_loss: 0.659

In [ ]:
# Trading simulation: only act on high-confidence buy signals
def get_fwd_returns_for_test(clean_dir, label_col, fwd_col):
    """Re-build test windows but keep the forward-return column instead of label."""
    all_fwd = []
    for path in sorted(clean_dir.glob("*.csv")):
        df = load_ticker_csv(path)
        df = add_technical_features(df)
        df = make_signal_labels(df)
        df = df.dropna(subset=FEATURES_T3 + [label_col, fwd_col])
        if len(df) < WINDOW_SIZE + 10: continue
        arr_d = df["Date"].values
        arr_f = df[fwd_col].values.astype(np.float32)
        for i in range(len(df) - WINDOW_SIZE):
            d = arr_d[i]
            if d >= cut_va_s:
                all_fwd.append(arr_f[i + WINDOW_SIZE - 1])
    return np.array(all_fwd)

In [ ]:
y_proba_sell = model_sell1.predict(X_te_sn, verbose=0).flatten()
evaluate_classifier(y_te_s, y_proba_sell, label="Task 3.2 — SELL classifier")

fwd_returns_sell = get_fwd_returns_for_test(VN_CLEAN_DIR, "sell_label", "fwd_ret_min")
# Note: for SELL, profitable PnL means SHORT positions — so PnL = -fwd_ret_min
trading_simulation(y_te_s, y_proba_sell, -fwd_returns_sell,
                   label="SELL (short)", threshold=0.6, cost_bps=20)


══════════ Task 3.2 — SELL classifier ══════════
  Test windows : 276,937
  Positive rate: 55.35%
  AUC          : 0.6616

   threshold     acc    prec   recall      F1    n_pos
        0.30  0.5949  0.5843   0.9289  0.7173   243677
        0.40  0.6190  0.6263   0.7729  0.6919   189165
        0.50  0.6108  0.6771   0.5673  0.6174   128425
        0.60  0.5669  0.7349   0.3401  0.4650    70927
        0.70  0.5052  0.8004   0.1413  0.2402    27054

  Confusion matrix (threshold = 0.50):
                 pred=0       pred=1
    true=0       82197       41469
    true=1       66315       86956

══════════ Trading sim (SELL (short)) — threshold 0.6 ══════════
  N trades   : 70,927  (25.61% of test windows)
  Win rate   : 92.47%
  Avg PnL    : +6.6430%
  Total PnL  : +471167.1875%  (sum of returns, not compounded)
  Sharpe (annualised, very rough): 18.23


{'n_trades': 70927,
 'win_rate': 0.9246972239062698,
 'avg_pnl': 0.06642987579107285,
 'total_pnl': 4711.671875,
 'sharpe': 18.229343508095013}

# Per-ticker breakdown

Some companies are inherently more predictable than others (lower volatility, clearer trends). Let's see which VNINDEX tickers our buy/sell signals work best on.

In [ ]:
def per_ticker_classification_report(
    y_true: np.ndarray,
    y_proba: np.ndarray,
    tickers: np.ndarray,
    threshold: float = 0.5,
    min_windows: int = 30,
    top_n: int = 15,
    label: str = "",
) -> pd.DataFrame:
    """Compute precision / recall / AUC per ticker. Useful for finding
    which companies the model handles best."""
    records = []
    for ticker in np.unique(tickers):
        mask = tickers == ticker
        if mask.sum() < min_windows:
            continue
        yt, yp_proba = y_true[mask], y_proba[mask]
        yp = (yp_proba >= threshold).astype(int)
        if len(np.unique(yt)) < 2:   # only one class present
            continue
        try: auc = roc_auc_score(yt, yp_proba)
        except: auc = float("nan")
        records.append({
            "ticker":    ticker,
            "n_windows": int(mask.sum()),
            "pos_rate":  float(yt.mean()),
            "precision": precision_score(yt, yp, zero_division=0),
            "recall":    recall_score   (yt, yp, zero_division=0),
            "f1":        f1_score       (yt, yp, zero_division=0),
            "auc":       auc,
        })
    df = pd.DataFrame(records).sort_values("auc", ascending=False)
    print(f"\n── Per-ticker classification: {label} ──")
    print(f"  Tickers analysed : {len(df)}")
    print(f"  Median AUC       : {df['auc'].median():.4f}")
    print(f"  Median F1        : {df['f1' ].median():.4f}")
    print(f"\n  Top {top_n} tickers by AUC:")
    print(df.head(top_n).to_string(index=False))
    return df

#summary_buy_per_ticker  = per_ticker_classification_report(y_te_b, y_proba_buy,  tickers_te_b, threshold=0.6, label="BUY")
summary_sell_per_ticker = per_ticker_classification_report(
    y_te_s, y_proba_sell, tickers_te_s, threshold=0.6, label="SELL")



── Per-ticker classification: SELL ──
  Tickers analysed : 348
  Median AUC       : 0.6267
  Median F1        : 0.4161

  Top 15 tickers by AUC:
     ticker  n_windows  pos_rate  precision   recall       f1      auc
SRF-History        803  0.689913   0.898305 0.574007 0.700441 0.790237
PJT-History        803  0.557908   0.816850 0.497768 0.618585 0.771548
EMC-History        647  0.578053   0.827839 0.604278 0.698609 0.765724
PGI-History        803  0.362391   0.671756 0.302405 0.417062 0.759524
CCI-History        802  0.617207   0.802899 0.559596 0.659524 0.756641
TYA-History        796  0.403266   0.778947 0.230530 0.355769 0.754511
AMD-History        801  0.596754   0.797619 0.560669 0.658477 0.750016
BBC-History        793  0.610340   0.856574 0.444215 0.585034 0.748917
BRC-History        769  0.553966   0.834320 0.330986 0.473950 0.740607
COM-History        794  0.701511   0.856777 0.601436 0.706751 0.734980
GTA-History        803  0.637609   0.847909 0.435547 0.575484 0.733865
KH